# PyTorch Tutorial: From Beginner to Deep Learning

This tutorial covers PyTorch from basic tensor operations to building, training, evaluating, and saving a neural network. You can copy each section into a Jupyter Notebook (`.ipynb`) and run the code cell by cell.

## Tutorial roadmap

1. Introduction to PyTorch
2. Tensors and tensor operations
3. Autograd and automatic differentiation
4. Building a neural network
5. Training a model with a real dataset
6. Evaluating model performance
7. Saving and loading a model
8. GPU training
9. Mini-project: Handwritten digit classification
10. Practice exercises

# 1. Introduction to PyTorch

### Text cell — What is PyTorch?

PyTorch is an open-source deep learning framework used to build and train machine learning models.

It provides:

- Tensors: Multidimensional arrays for numerical computation.
- Autograd: Automatic calculation of gradients.
- Neural network modules: Layers and components for building models.
- Optimizers: Algorithms that update model parameters during training.
- GPU support: Acceleration on compatible hardware.

Common applications include image recognition, natural language processing, generative AI, and time-series forecasting.

### Code cell — Install and import PyTorch

In [ ]:
# Run this cell only if PyTorch is not installed.
# In Jupyter Notebook, uncomment the following line:
# %pip install torch torchvision matplotlib scikit-learn


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Expected output: The PyTorch version and whether a compatible CUDA GPU is available.

# 2. Understanding Tensors

### Text cell — What is a tensor?

A tensor is a generalization of a scalar, vector, and matrix to multiple dimensions.

| Type | Example | Dimensions |
| --- | --- | --- |
| Scalar | `5` | 0D |
| Vector | `[1, 2, 3]` | 1D |
| Matrix | `[[1, 2], [3, 4]]` | 2D |
| Image batch | Multiple images | Usually 4D |

PyTorch tensors are similar to NumPy arrays, but they also support GPU computation and automatic differentiation.

### Code cell — Create tensors

In [ ]:
import torch

# Scalar
scalar = torch.tensor(5)

# Vector
vector = torch.tensor([1, 2, 3])

# Matrix
matrix = torch.tensor([
    [1, 2],
    [3, 4]
])

# Random tensor
random_tensor = torch.rand(3, 4)

print("Scalar:", scalar)
print("Vector:", vector)
print("Matrix:\n", matrix)
print("Random tensor:\n", random_tensor)

print("Matrix shape:", matrix.shape)
print("Matrix dimensions:", matrix.ndim)
print("Matrix data type:", matrix.dtype)


### Code cell — Tensor operations

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Addition:", a + b)
print("Subtraction:", a - b)
print("Multiplication:", a * b)
print("Division:", a / b)

# Matrix multiplication
x = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

y = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

print("Matrix multiplication:\n", x @ y)


### Code cell — Reshape tensors

In [ ]:
x = torch.arange(12)

print("Original:", x)

reshaped = x.reshape(3, 4)

print("Reshaped:\n", reshaped)
print("Shape:", reshaped.shape)


Key concept: Tensor shapes must be compatible with the operation being performed.

# 3. Automatic Differentiation with Autograd

### Text cell — What is autograd?

Training a neural network requires calculating how much each parameter contributes to the prediction error.

PyTorch's autograd system automatically calculates derivatives using computational graphs.

For example, suppose:

$$
y=x^2+3x+2
$$

The derivative is:

$$
\frac{dy}{dx}=2x+3
$$

At $x=2$, the derivative is $7$.

### Code cell — Calculate a gradient

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)

y = x**2 + 3*x + 2

y.backward()

print("Output:", y.item())
print("Gradient:", x.grad.item())


Expected output:

In [ ]:
Output: 12.0
Gradient: 7.0


### Text cell — Why are gradients important?

Gradients indicate how the loss changes when model parameters change. Optimizers use these gradients to adjust the parameters and reduce prediction error.

### Code cell — Gradient descent example

In [ ]:
import torch

# Start with an incorrect parameter
w = torch.tensor(0.0, requires_grad=True)

# Target: minimize (w - 5)^2
learning_rate = 0.1

for epoch in range(50):
    loss = (w - 5)**2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad

    w.grad.zero_()

print("Learned parameter:", w.item())
print("Final loss:", loss.item())


The learned parameter should approach `5`.

Important: `backward()` accumulates gradients. In a training loop, gradients are usually cleared before the next backward pass.

# 4. Build Your First Neural Network

### Text cell — Neural network architecture

A neural network consists of interconnected computational layers.

For this example:

- Input: Two numerical features.
- Hidden layer: Eight neurons with ReLU activation.
- Output: One numerical prediction.

The network learns a nonlinear relationship between the input and target.

### Code cell — Define the model

In [ ]:
import torch
import torch.nn as nn

class SimpleNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

model = SimpleNeuralNetwork()

print(model)


### Text cell — Understanding the layers

- `nn.Linear(2, 8)` maps two input features to eight hidden units.
- `nn.ReLU()` introduces nonlinearity.
- `nn.Linear(8, 1)` generates one output.
- `forward()` defines how input data moves through the network.

PyTorch automatically registers the trainable parameters in the model.

### Code cell — Make a prediction

In [ ]:
sample = torch.tensor([[0.3, 0.8]])

prediction = model(sample)

print("Prediction:", prediction.item())


The output will initially be arbitrary because the model has not been trained.

# 5. Train a Neural Network on Synthetic Data

### Text cell — Define the problem

We will train a binary classifier using two numerical features.

The target is defined by this rule:

$$
y =
\begin{cases}
1 & \text{if } x_1+x_2 \geq 1\\
0 & \text{otherwise}
\end{cases}
$$

The neural network will learn to predict the target from examples.

### Code cell — Generate a dataset

In [ ]:
import torch

torch.manual_seed(42)

# Generate 1,000 examples with two features each
X = torch.rand(1000, 2)

# Generate binary labels
y = (X.sum(dim=1) >= 1.0).float().unsqueeze(1)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("First five inputs:\n", X[:5])
print("First five labels:\n", y[:5])


### Text cell — Split the dataset

We use separate training and testing datasets so that evaluation measures performance on examples the model did not train on.

### Code cell — Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X.numpy(),
    y.numpy(),
    test_size=0.2,
    random_state=42,
    stratify=y.numpy()
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


### Text cell — Configure the model

For binary classification, we will use:

- `nn.Linear`: Learnable layers.
- `nn.ReLU`: Hidden-layer activation.
- `nn.BCEWithLogitsLoss`: Binary classification loss.
- `optim.Adam`: Optimizer for updating model parameters.

`BCEWithLogitsLoss` combines sigmoid and binary cross-entropy in a numerically stable operation. Therefore, the model's final layer should output raw logits rather than applying sigmoid inside the model.

### Code cell — Build the training model

In [ ]:
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

print(model)


### Code cell — Training loop

In [ ]:
epochs = 100
loss_history = []

for epoch in range(epochs):
    # Forward pass
    logits = model(X_train)

    # Calculate loss
    loss = criterion(logits, y_train)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )


### Text cell — What happens during training?

Each epoch performs five operations:

1. The model generates predictions.
2. The loss function measures prediction error.
3. `zero_grad()` clears gradients from the previous update.
4. `backward()` calculates gradients.
5. `optimizer.step()` updates the parameters.

As training progresses, the loss should generally decrease, although individual updates may fluctuate.

### Code cell — Plot the training loss

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True)

plt.show()


# 6. Evaluate the Model

### Text cell — Why evaluate?

A model can perform well on its training examples but poorly on unseen data. Testing helps estimate whether it has learned a useful relationship.

For binary classification, we can calculate accuracy, precision, recall, and a confusion matrix.

### Code cell — Calculate test accuracy

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model.eval()

with torch.no_grad():
    test_logits = model(X_test)

    test_probabilities = torch.sigmoid(test_logits)

    test_predictions = (
        test_probabilities >= 0.5
    ).int().numpy().ravel()

y_true = y_test.numpy().ravel().astype(int)

accuracy = accuracy_score(
    y_true,
    test_predictions
)

print("Test accuracy:", round(accuracy, 4))

print("\nClassification report:")
print(classification_report(y_true, test_predictions))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, test_predictions))


### Text cell — Important evaluation concepts

- Accuracy: Fraction of predictions that are correct.
- Precision: Fraction of predicted positives that are actual positives.
- Recall: Fraction of actual positives that are correctly identified.
- Confusion matrix: Counts correct and incorrect predictions for each class.

For imbalanced datasets, accuracy alone may be misleading.

### Code cell — Predict a new example

In [ ]:
sample = torch.tensor([[0.3, 0.8]])

model.eval()

with torch.no_grad():
    logit = model(sample)
    probability = torch.sigmoid(logit).item()

predicted_class = int(probability >= 0.5)

print("Probability of class 1:", round(probability, 4))
print("Predicted class:", predicted_class)


Since the input features sum to more than one, the expected class under our synthetic labeling rule is `1`.

# 7. Save and Load the Model

### Text cell — Why save a model?

Training may take time. Saving the learned parameters allows you to reuse the model later without training it again.

PyTorch commonly saves a model's `state_dict`, which contains its learned parameters.

### Code cell — Save model parameters

In [ ]:
torch.save(
    model.state_dict(),
    "simple_nn.pth"
)

print("Model parameters saved.")


### Code cell — Load model parameters

In [ ]:
loaded_model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

loaded_model.load_state_dict(
    torch.load(
        "simple_nn.pth",
        map_location="cpu",
        weights_only=True
    )
)

loaded_model.eval()

sample = torch.tensor([[0.3, 0.8]])

with torch.no_grad():
    probability = torch.sigmoid(
        loaded_model(sample)
    ).item()

print("Loaded model probability:", probability)


Note: The architecture used when loading must match the architecture used when saving. Keep the model definition and any preprocessing steps with your project.

# 8. Run PyTorch on a GPU

### Text cell — CPU vs. GPU

PyTorch can use a compatible CUDA GPU for many operations. A GPU can accelerate training, especially for larger models and datasets.

Not every computer has a compatible GPU, so the following code automatically selects CUDA when available and otherwise uses the CPU.

### Code cell — Select the device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


### Code cell — Move the model and data

In [ ]:
model = model.to(device)

X_train_device = X_train.to(device)
y_train_device = y_train.to(device)

X_test_device = X_test.to(device)
y_test_device = y_test.to(device)

print("Model device:", next(model.parameters()).device)
print("Training data device:", X_train_device.device)


When training on the selected device, the model and tensors involved in the same operation must be on compatible devices.

For a small dataset such as this one, a GPU may not improve speed because the overhead of GPU execution can outweigh the computational benefit.

# 9. Mini-Project: Handwritten Digit Classification

### Text cell — Project objective

Build a neural network that recognizes handwritten digits from the MNIST dataset.

Each image is a 28 × 28 grayscale image, and each label represents one of ten digits, from 0 to 9.

We will use `torchvision` to download the dataset.

### Code cell — Load the MNIST dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

### Text cell — Visualize sample images

### Code cell

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()


### Text cell — Build the digit classifier

The network will flatten each image into 784 input values, pass them through a hidden layer, and output ten class scores.

We will use cross-entropy loss, which expects raw class scores called logits and integer labels.

### Code cell — Define the model

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

digit_model = DigitClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    digit_model.parameters(),
    lr=0.001
)

print(digit_model)


### Code cell — Train the digit classifier

In [ ]:
epochs = 5

for epoch in range(epochs):
    digit_model.train()

    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        logits = digit_model(images)

        loss = criterion(logits, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    average_loss = running_loss / len(train_dataset)

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )


### Text cell — Evaluate the digit classifier

### Code cell

In [ ]:
digit_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = digit_model(images)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test accuracy percentage: {accuracy * 100:.2f}%")


The actual accuracy depends on training and hardware execution, but this simple model should learn to recognize many handwritten digits.

### Code cell — Display a prediction

In [ ]:
digit_model.eval()

image, label = test_dataset[0]

with torch.no_grad():
    logits = digit_model(
        image.unsqueeze(0).to(device)
    )

    prediction = logits.argmax(dim=1).item()

plt.imshow(image.squeeze(), cmap="gray")
plt.title(
    f"Actual: {label} | Predicted: {prediction}"
)
plt.axis("off")
plt.show()


# 10. Practice Exercises

Complete these exercises after running the tutorial.

1. Create a tensor of shape `(5, 5)` and calculate its mean.
2. Change the neural network's hidden layer from 8 neurons to 32 neurons.
3. Compare ReLU and sigmoid in the hidden layer.
4. Train the binary classifier with different learning rates.
5. Plot the training loss for 50 and 100 epochs.
6. Save the model and verify that the loaded model makes the same predictions.
7. Modify the MNIST model to use two hidden layers.
8. Compare the fully connected MNIST model with a convolutional neural network.
9. Calculate the confusion matrix for MNIST predictions.
10. Run the training code on a GPU if one is available.

# 11. Interview Questions

### Theoretical Questions

1. What is PyTorch?
2. What is a tensor?
3. What is automatic differentiation?
4. What does `requires_grad=True` mean?
5. What is the purpose of `backward()`?
6. Why do we use `optimizer.zero_grad()`?
7. What is the difference between `model.train()` and `model.eval()`?
8. What is the difference between `torch.no_grad()` and gradient tracking?
9. Why is `BCEWithLogitsLoss` used for binary classification?
10. Why does `CrossEntropyLoss` expect logits rather than softmax probabilities?

### Practical Questions

1. Build a neural network using `nn.Sequential`.
2. Train a binary classifier using PyTorch.
3. Explain the forward and backward passes.
4. Save and load a `state_dict`.
5. Move a model to a GPU.
6. Evaluate a classifier using accuracy and a confusion matrix.
7. Train a digit classifier using MNIST.
8. Explain how batch size affects training.
9. Compare Adam and stochastic gradient descent.
10. Explain how you would prevent overfitting.

# 12\. Final Summary

By completing this tutorial, you will have learned how to:

- Create and manipulate PyTorch tensors.
- Calculate gradients using autograd.
- Define a neural network with `nn.Module`.
- Train a model using a loss function and optimizer.
- Evaluate predictions on unseen data.
- Save and reload model parameters.
- Use a GPU when compatible hardware is available.
- Build an end-to-end handwritten digit classifier.

Recommended next tutorial: PyTorch CNNs for image classification, followed by transfer learning and a practical Generative AI project.